In [ ]:
!pip install openai
!pip install langchain
!pip install tqdm
!pip install chromadb
!pip install tiktoken
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.5/325.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.6/974.6 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.8/321.8 kB 30.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 17.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.5/559.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━

In [ ]:
# key 설정

import os
import openai

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# rag 위한 파일 준비

import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hwchase17/chat-your-data/master/state_of_the_union.txt",
    filename="state_of_the_union.txt"
)

('state_of_the_union.txt', <http.client.HTTPMessage at 0x7beac2e5f6d0>)

In [ ]:
from tqdm import tqdm

Simple TextLoader 구현해 보기

In [ ]:
class SimpleTextLoader:

  def __init__(self, file_path):
      self.file_path = file_path

  def load(self):
      text = ''   # 파일에서 읽은 데이터 저장하는 변수
      with open(self.file_path, 'r', encoding='utf-8') as f:
        text = f.read()
      return text

SimpleCharacterTextSplitter 구현

In [ ]:
# 이삿짐 센터
# 여러가지 box size
# box 이사 전에 가져다 줌
# 짐을 박스에 담는다.

class SimpleCharacterTextSplitter:

    def __init__(self, chunk_size, chunk_overlap, separator_pattern='\n\n'):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separator_pattern = separator_pattern

    # 문장 분할 함수(메소드)
    def split_document(self, documents):

        # 파일 전체 내용 >> 문단 단위로 나누기 (기준: separator_pattern)
        splits = documents.split(self.separator_pattern)

        chunks = []  # 최종적으로 생성될 chunks 저장
        current_chunk = splits[0]
        # 첫 번째 문단 >> 초기 chunk 로 설정

        for split in tqdm(splits[1:], desc="splitting..."):
            # current chunk(현재 청크)하고 다음 문단, 그리고 구분자가  chunk_size를 초과되는 지 확인
            if len(current_chunk) + len(split) + len(self.separator_pattern) > self.chunk_size:
               chunks.append(current_chunk.strip())
               current_chunk = split  # 새로운 청크의 시작
            else:
                # 초과하지 않으면 >> 현재 청크에 구분자 + 다음 문단 추가
                current_chunk += self.separator_pattern


        # 마지막 청크 추가
        if current_chunk:
            chunks.append(current_chunk.strip())

        return chunks

        # A split 500자
        # B split 10자
        # C split 600자

        # 위 split들을 청크(박스)에 담는 행위
        # A + B = 510 첫번째 박스
        # C는 600자라서 A + B + C = 1110 > 1000 박스 크기 초과
        # C 두번째 박스
        # 첫번째 박스 (510 : A, B)
        # 두번째 박스 (600 : C)


SimpleOpenAIEmbeddings 구현해보기

In [ ]:
from openai import OpenAI

class SimpleOpenAIEmbeddings:

    def embed_query(self, text):
        client = OpenAI()
        response = client.embeddings.create(
                  input=text,
                  model='text-embedding-ada-002'
        )
        return response.data[0].embedding

SimpleVectorStore 구현

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class SimpleVectorStore:

    def __init__(self, docs, embedding):
        self.embedding = embedding
        self.documents = []   # 문서 내용을 저장할 리스트
        self.vectors = []     # 문서의 벡터 저장할 리스트

        # 모든 문서에 대해서 임베딩 수행 >> 벡터, 문서내용 저장
        for doc in tqdm(docs, desc='embeding...'):
            self.documents.append(doc)
            vector = self.embedding.embed_query(doc) # 문서 >> 벡터 변환
            self.vectors.append(vector)

    # 유사도 검색 메소드
    def similarity_search(self, query, k=4):   # 가장 유사한 4개 문서 반환
         query_vector = self.embedding.embed_query(query)  # 쿼리 >> 벡터 변환

         if not self.vectors: # 저장된 벡터 없으면 >> 빈 리스트 반환
            return []

         similarities = cosine_similarity([query_vector], self.vectors)[0]
         # 쿼리 벡터 - 저장된 모든 벡터 간 코사인 유사도 계산
         sorted_doc_similarities = sorted(zip(self.documents, similarities), key=lambda x: x[1], reverse=True)
         # zip : (문서, 유사도) 튜플 형태로 묶어줌, 유사도의 내림차순 정렬
         return sorted_doc_similarities[:k] # 상위 k 개 문서 반환

    def as_retriever(self, k=4):
        return SimpleRetriever(self, k)

SimpleRetriever 구현하기

In [ ]:
class SimpleRetriever:
    def __init__(self, vector_store, k=4):
        self.vector_store = vector_store
        self.k = k

    def get_relevant_documents(self, query):
        docs = self.vector_store.similarity_search(query, self.k)
        return docs

In [ ]:
raw_documents = SimpleTextLoader('state_of_the_union.txt').load()
text_splitter = SimpleCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)

db = SimpleVectorStore(documents, SimpleOpenAIEmbeddings())

embeding...: 100%|██████████| 42/42 [00:13<00:00,  3.14it/s]


In [ ]:
len(documents)

42

In [ ]:
documents[0:4]

['Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.',
 'Groups of citizens blocking tanks with

In [ ]:
query = "What did the president say about Ketanji Brown Jackson"

In [ ]:
docs = db.similarity_search(query)
docs

[('Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.',
  0.8152000921997997),
 ('A former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. Since she’

In [ ]:
print(docs[0][0])

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.


한글 벡터스토어

In [ ]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/puzzlet/constitution-kr/master/%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD%20%ED%97%8C%EB%B2%95.txt",
    filename="korea_constitution.txt"
)

('korea_constitution.txt', <http.client.HTTPMessage at 0x7beabbe15990>)

In [ ]:
raw_documents = SimpleTextLoader('korea_constitution.txt').load()
text_splitter = SimpleCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)

splitting...: 100%|██████████| 152/152 [00:00<00:00, 379258.90it/s]


In [ ]:
documents

['전문\n\n유구한 역사와 전통에 빛나는 우리 대한국민은 3·1운동으로 건립된 대한민국임시정부의 법통과 불의에 항거한 4·19민주이념을 계승하고, 조국의 민주개혁과 평화적 통일의 사명에 입각하여 정의·인도와 동포애로써 민족의 단결을 공고히 하고, 모든 사회적 폐습과 불의를 타파하며, 자율과 조화를 바탕으로 자유민주적 기본질서를 더욱 확고히 하여 정치·경제·사회·문화의 모든 영역에 있어서 각인의 기회를 균등히 하고, 능력을 최고도로 발휘하게 하며, 자유와 권리에 따르는 책임과 의무를 완수하게 하여, 안으로는 국민생활의 균등한 향상을 기하고 밖으로는 항구적인 세계평화와 인류공영에 이바지함으로써 우리들과 우리들의 자손의 안전과 자유와 행복을 영원히 확보할 것을 다짐하면서 1948년 7월 12일에 제정되고 8차에 걸쳐 개정된 헌법을 이제 국회의 의결을 거쳐 국민투표에 의하여 개정한다.\n\n제1장 총강\n\n제1조 ① 대한민국은 민주공화국이다.\n② 대한민국의 주권은 국민에게 있고, 모든 권력은 국민으로부터 나온다.\n\n제2조 ① 대한민국의 국민이 되는 요건은 법률로 정한다.\n② 국가는 법률이 정하는 바에 의하여 재외국민을 보호할 의무를 진다.\n\n제3조 대한민국의 영토는 한반도와 그 부속도서로 한다.\n\n제4조 대한민국은 통일을 지향하며, 자유민주적 기본질서에 입각한 평화적 통일 정책을 수립하고 이를 추진한다.\n\n제5조 ① 대한민국은 국제평화의 유지에 노력하고 침략적 전쟁을 부인한다.\n② 국군은 국가의 안전보장과 국토방위의 신성한 의무를 수행함을 사명으로 하며, 그 정치적 중립성은 준수된다.\n\n제6조 ① 헌법에 의하여 체결·공포된 조약과 일반적으로 승인된 국제법규는 국내법과 같은 효력을 가진다.\n② 외국인은 국제법과 조약이 정하는 바에 의하여 그 지위가 보장된다.\n\n제7조 ① 공무원은 국민전체에 대한 봉사자이며, 국민에 대하여 책임을 진다.\n② 공무원의 신분과 정치적 중립성은 법률이 정하는 바에 의하여 보장된다.',
 '제8조 ① 정당의 설

In [ ]:
db = SimpleVectorStore(documents, SimpleOpenAIEmbeddings())

embeding...: 100%|██████████| 22/22 [00:06<00:00,  3.55it/s]


In [ ]:
query = "대한민국의 대통령 임기는?"
docs = db.similarity_search(query)
docs

[('제65조 ① 대통령·국무총리·국무위원·행정각부의 장·헌법재판소 재판관·법관·중앙선거관리위원회 위원·감사원장·감사위원 기타 법률이 정한 공무원이 그 직무집행에 있어서 헌법이나 법률을 위배한 때에는 국회는 탄핵의 소추를 의결할 수 있다.\n② 제1항의 탄핵소추는 국회재적의원 3분의 1 이상의 발의가 있어야 하며, 그 의결은 국회재적의원 과반수의 찬성이 있어야 한다. 다만, 대통령에 대한 탄핵소추는 국회재적의원 과반수의 발의와 국회재적의원 3분의 2 이상의 찬성이 있어야 한다.\n③ 탄핵소추의 의결을 받은 자는 탄핵심판이 있을 때까지 그 권한행사가 정지된다.\n④ 탄핵결정은 공직으로부터 파면함에 그친다. 그러나, 이에 의하여 민사상이나 형사상의 책임이 면제되지는 아니한다.\n\n제4장 정부\n제1절 대통령\n\n제66조 ① 대통령은 국가의 원수이며, 외국에 대하여 국가를 대표한다.\n② 대통령은 국가의 독립·영토의 보전·국가의 계속성과 헌법을 수호할 책무를 진다.\n③ 대통령은 조국의 평화적 통일을 위한 성실한 의무를 진다.\n④ 행정권은 대통령을 수반으로 하는 정부에 속한다.\n\n제67조 ① 대통령은 국민의 보통·평등·직접·비밀선거에 의하여 선출한다.\n② 제1항의 선거에 있어서 최고득표자가 2인 이상인 때에는 국회의 재적의원 과반수가 출석한 공개회의에서 다수표를 얻은 자를 당선자로 한다.\n③ 대통령후보자가 1인일 때에는 그 득표수가 선거권자 총수의 3분의 1 이상이 아니면 대통령으로 당선될 수 없다.\n④ 대통령으로 선거될 수 있는 자는 국회의원의 피선거권이 있고 선거일 현재 40세에 달하여야 한다.\n⑤ 대통령의 선거에 관한 사항은 법률로 정한다.\n\n제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.',
  0.8489666521692316),
 ('제76조 ① 대통령은

벡터스토어 튜닝하기

In [ ]:
raw_documents = SimpleTextLoader('korea_constitution.txt').load()
text_splitter = SimpleCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)

splitting...: 100%|██████████| 152/152 [00:00<00:00, 67549.71it/s]


In [ ]:
documents[:10]

['전문',
 '유구한 역사와 전통에 빛나는 우리 대한국민은 3·1운동으로 건립된 대한민국임시정부의 법통과 불의에 항거한 4·19민주이념을 계승하고, 조국의 민주개혁과 평화적 통일의 사명에 입각하여 정의·인도와 동포애로써 민족의 단결을 공고히 하고, 모든 사회적 폐습과 불의를 타파하며, 자율과 조화를 바탕으로 자유민주적 기본질서를 더욱 확고히 하여 정치·경제·사회·문화의 모든 영역에 있어서 각인의 기회를 균등히 하고, 능력을 최고도로 발휘하게 하며, 자유와 권리에 따르는 책임과 의무를 완수하게 하여, 안으로는 국민생활의 균등한 향상을 기하고 밖으로는 항구적인 세계평화와 인류공영에 이바지함으로써 우리들과 우리들의 자손의 안전과 자유와 행복을 영원히 확보할 것을 다짐하면서 1948년 7월 12일에 제정되고 8차에 걸쳐 개정된 헌법을 이제 국회의 의결을 거쳐 국민투표에 의하여 개정한다.',
 '제1장 총강\n\n제1조 ① 대한민국은 민주공화국이다.\n② 대한민국의 주권은 국민에게 있고, 모든 권력은 국민으로부터 나온다.',
 '제2조 ① 대한민국의 국민이 되는 요건은 법률로 정한다.\n② 국가는 법률이 정하는 바에 의하여 재외국민을 보호할 의무를 진다.',
 '제3조 대한민국의 영토는 한반도와 그 부속도서로 한다.\n\n제4조 대한민국은 통일을 지향하며, 자유민주적 기본질서에 입각한 평화적 통일 정책을 수립하고 이를 추진한다.',
 '제5조 ① 대한민국은 국제평화의 유지에 노력하고 침략적 전쟁을 부인한다.\n② 국군은 국가의 안전보장과 국토방위의 신성한 의무를 수행함을 사명으로 하며, 그 정치적 중립성은 준수된다.',
 '제6조 ① 헌법에 의하여 체결·공포된 조약과 일반적으로 승인된 국제법규는 국내법과 같은 효력을 가진다.\n② 외국인은 국제법과 조약이 정하는 바에 의하여 그 지위가 보장된다.',
 '제7조 ① 공무원은 국민전체에 대한 봉사자이며, 국민에 대하여 책임을 진다.\n② 공무원의 신분과 정치적 중립성은 법률이 정하는 바에 의하여 보장된다.',
 '제8조 

In [ ]:
db = SimpleVectorStore(documents, SimpleOpenAIEmbeddings())

embeding...: 100%|██████████| 137/137 [00:37<00:00,  3.68it/s]


In [ ]:
query = "대한민국의 대통령 임기는 몇 년인가?"
docs = db.similarity_search(query)
docs

[('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', 0.8788696183118142),
 ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.',
  0.8646525023381494),
 ('제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.\n② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n④ 법관의 정년은 법률로 정한다.',
  0.8545891190873053),
 ('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."',
  0.8502869747507775)]

검색기 만들기

In [ ]:
retriever = db.as_retriever()

In [ ]:
query ='대통령의 임기는 몇 년인가?'

In [ ]:
unique_docs = retriever.get_relevant_documents(query)
unique_docs

[('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', 0.880359162000643),
 ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.',
  0.8672309105052376),
 ('제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.\n② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n④ 법관의 정년은 법률로 정한다.',
  0.8548423908185656),
 ('제98조 ① 감사원은 원장을 포함한 5인 이상 11인 이하의 감사위원으로 구성한다.\n② 원장은 국회의 동의를 얻어 대통령이 임명하고, 그 임기는 4년으로 하며, 1차에 한하여 중임할 수 있다.\n③ 감사위원은 원장의 제청으로 대통령이 임명하고, 그 임기는 4년으로 하며, 1차에 한하여 중임할 수 있다.',
  0.850684521967966)]

챗봇 만들기

In [ ]:
import openai

system_prompt_template = ("You are a helpful assistant. "
                          "Based on the following content, "
                          "kindly and comprehensively respond to user questions. write in Korean."
                          "[Content]"
                          "{content}"
                          "")

# 검색 기반 QA 클래스 정의
class SimpleRetrievalQA():

  def __init__(self, retriever):
    self.retriever = retriever
    # 검색기를 인스턴스 변수로 저장

  def invoke(self, query):
    docs = self.retriever.get_relevant_documents(query)
    print(docs)
    # 질의에 맞는 문서 검색

    # 검색된 문서 출력
    for i, doc in enumerate(docs):
      print("[#" +str(i)+ "]", doc[1]) # 문서의 내용
      print(doc[0]) # 문서의 유사도

    completion = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_prompt_template.format(content=docs)},
            # 시스템 메시지: 검색된 문서 내용 >> 시스템 프롬프트에 전달
            {"role": "user", "content": query},]

    )
    return completion.choices[0].message.content

In [ ]:
chain = SimpleRetrievalQA(retriever)

answer = chain.invoke("대통령의 임기는?")
print('>>', answer)

[('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.', 0.8629858682823626), ('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."', 0.8561573009540799), ('제4장 정부\n제1절 대통령', 0.8561026328257411), ('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', 0.8539171368087439)]
[#0] 0.8629858682823626
제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.
② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.
[#1] 0.8561573009540799
제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.
"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."
[#2] 0.8561026328257411
제4장 정부
제1절 대통령
[#3] 0.8539171368087439
제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
>> 대통령의 임기는 5년으로 하며, 중임할 수 없습니다. (제70조)


In [ ]:
chain = SimpleRetrievalQA(retriever)

answer = chain.invoke("대통령은 중임을 할 수 있나?")
print('>>', answer)

[('제72조 대통령은 필요하다고 인정할 때에는 외교·국방·통일 기타 국가안위에 관한 중요정책을 국민투표에 붙일 수 있다.', 0.8697810330275314), ('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."', 0.8659525498290721), ('제76조 ① 대통령은 내우·외환·천재·지변 또는 중대한 재정·경제상의 위기에 있어서 국가의 안전보장 또는 공공의 안녕질서를 유지하기 위하여 긴급한 조치가 필요하고 국회의 집회를 기다릴 여유가 없을 때에 한하여 최소한으로 필요한 재정·경제상의 처분을 하거나 이에 관하여 법률의 효력을 가지는 명령을 발할 수 있다.\n② 대통령은 국가의 안위에 관계되는 중대한 교전상태에 있어서 국가를 보위하기 위하여 긴급한 조치가 필요하고 국회의 집회가 불가능한 때에 한하여 법률의 효력을 가지는 명령을 발할 수 있다.\n③ 대통령은 제1항과 제2항의 처분 또는 명령을 한 때에는 지체없이 국회에 보고하여 그 승인을 얻어야 한다.\n④ 제3항의 승인을 얻지 못한 때에는 그 처분 또는 명령은 그때부터 효력을 상실한다. 이 경우 그 명령에 의하여 개정 또는 폐지되었던 법률은 그 명령이 승인을 얻지 못한 때부터 당연히 효력을 회복한다.\n⑤ 대통령은 제3항과 제4항의 사유를 지체없이 공포하여야 한다.', 0.8654103312658407), ('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', 0.8606249934003996)]
[#0] 0.8697810330275314
제72조 대통령은 필요하다고 인정할 때에는 외교·국방·통일 기타 국가안위에 관한 중요정책을 국민투표에 붙일 수 있다.
[#1] 0.8659525498290721
제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.
"나는 헌법을 준수하고 국가를 보위하며 

In [ ]:
def chat_with_user(user_message):
   ai_message = chain.invoke(user_message)
   return ai_message

while True:
   user_message = input("user> ")
   if user_message.lower() == "quit":
      break
   ai_message = chat_with_user(user_message)
   print(f"ai >> {ai_message}")

user> 대통령은 두 번 당선 가능해?
[('제67조 ① 대통령은 국민의 보통·평등·직접·비밀선거에 의하여 선출한다.\n② 제1항의 선거에 있어서 최고득표자가 2인 이상인 때에는 국회의 재적의원 과반수가 출석한 공개회의에서 다수표를 얻은 자를 당선자로 한다.\n③ 대통령후보자가 1인일 때에는 그 득표수가 선거권자 총수의 3분의 1 이상이 아니면 대통령으로 당선될 수 없다.\n④ 대통령으로 선거될 수 있는 자는 국회의원의 피선거권이 있고 선거일 현재 40세에 달하여야 한다.\n⑤ 대통령의 선거에 관한 사항은 법률로 정한다.', 0.8642964778233577), ('제72조 대통령은 필요하다고 인정할 때에는 외교·국방·통일 기타 국가안위에 관한 중요정책을 국민투표에 붙일 수 있다.', 0.85292712816032), ('제48조 국회는 의장 1인과 부의장 2인을 선출한다.', 0.8522011065887565), ('제2조 ① 이 헌법에 의한 최초의 대통령선거는 이 헌법시행일 40일 전까지 실시한다.\n② 이 헌법에 의한 최초의 대통령의 임기는 이 헌법시행일로부터 개시한다.', 0.8513580976039927)]
[#0] 0.8642964778233577
제67조 ① 대통령은 국민의 보통·평등·직접·비밀선거에 의하여 선출한다.
② 제1항의 선거에 있어서 최고득표자가 2인 이상인 때에는 국회의 재적의원 과반수가 출석한 공개회의에서 다수표를 얻은 자를 당선자로 한다.
③ 대통령후보자가 1인일 때에는 그 득표수가 선거권자 총수의 3분의 1 이상이 아니면 대통령으로 당선될 수 없다.
④ 대통령으로 선거될 수 있는 자는 국회의원의 피선거권이 있고 선거일 현재 40세에 달하여야 한다.
⑤ 대통령의 선거에 관한 사항은 법률로 정한다.
[#1] 0.85292712816032
제72조 대통령은 필요하다고 인정할 때에는 외교·국방·통일 기타 국가안위에 관한 중요정책을 국민투표에 붙일 수 있다.
[#2] 0.8522011065887565
제48조 국회는 의장 1인과 부의장 

In [ ]:
retriever = db.as_retriever(k=3)
chain = SimpleRetrievalQA(retriever)

answer = chain.invoke("대통령의 임기는?")
print('>>', answer)

[('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.', 0.8629858682823626), ('제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.\n"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."', 0.8561573009540799), ('제4장 정부\n제1절 대통령', 0.8561026328257411)]
[#0] 0.8629858682823626
제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.
② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.
[#1] 0.8561573009540799
제69조 대통령은 취임에 즈음하여 다음의 선서를 한다.
"나는 헌법을 준수하고 국가를 보위하며 조국의 평화적 통일과 국민의 자유와 복리의 증진 및 민족문화의 창달에 노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다."
[#2] 0.8561026328257411
제4장 정부
제1절 대통령
>> 대통령의 임기는 5년이며, 만료되는 때에는 임기만료 70일 내지 40일 전에 후임자를 선거합니다.


Local Embedding Model

In [ ]:
!pip install -U langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 8.3 MB/s eta 0:00:00


In [ ]:
import langchain
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
raw_documents = SimpleTextLoader('korea_constitution.txt').load()
text_splitter = SimpleCharacterTextSplitter(chunk_size=10, chunk_overlap=0)
documents = text_splitter.split_document(raw_documents)

embed_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-sts")
print(embed_model)

db = SimpleVectorStore(documents, embed_model)

In [ ]:
query = "대통령의 임기는?"
docs = db.similarity_search(query)
docs

[('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', 0.5381746278316393),
 ('제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.\n② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n④ 법관의 정년은 법률로 정한다.',
  0.4657673848386544),
 ('제42조 국회의원의 임기는 4년으로 한다.', 0.4376108557468448),
 ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.',
  0.43043348472810733)]

In [ ]:
retriever = db.as_retriever()
chain = SimpleRetrievalQA(retriever)
answer = chain.invoke("대통령의 임기는?")

print(">> ", answer)

[('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', 0.5381746278316393), ('제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.\n② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.\n④ 법관의 정년은 법률로 정한다.', 0.4657673848386544), ('제42조 국회의원의 임기는 4년으로 한다.', 0.4376108557468448), ('제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.\n② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.', 0.43043348472810733)]
[#0] 0.5381746278316393
제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
[#1] 0.4657673848386544
제105조 ① 대법원장의 임기는 6년으로 하며, 중임할 수 없다.
② 대법관의 임기는 6년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.
③ 대법원장과 대법관이 아닌 법관의 임기는 10년으로 하며, 법률이 정하는 바에 의하여 연임할 수 있다.
④ 법관의 정년은 법률로 정한다.
[#2] 0.4376108557468448
제42조 국회의원의 임기는 4년으로 한다.
[#3] 0.43043348472810733
제68조 ① 대통령의 임기가 만료되는 때에는 임기만료 70일 내지 40일전에 후임자를 선거한다.
② 대통령이 궐위된 때 또는 대통령 당선자가 사망하거나 판결 기타의 사유로 그 자격을 상실한 때에는 60일 이내에 후임자를 선거한다.
>>  대통령의 임기는 5년으로 하며, 중임할 수 없습니다.


In [ ]:
def chat_with_user(user_message):
   ai_message = chain.invoke(user_message)
   return ai_message

while True:
   user_message = input("user> ")
   if user_message.lower() == "quit":
      break
   ai_message = chat_with_user(user_message)
   print(f"ai >> {ai_message}")

user> 닥터윌이 어떻게 하면 대통령이 될 수 있을끼?
[('제94조 행정각부의 장은 국무위원 중에서 국무총리의 제청으로 대통령이 임명한다.', 0.3535967762969585), ('제86조 ① 국무총리는 국회의 동의를 얻어 대통령이 임명한다.\n② 국무총리는 대통령을 보좌하며, 행정에 관하여 대통령의 명을 받아 행정각부를 통할한다.\n③ 군인은 현역을 면한 후가 아니면 국무총리로 임명될 수 없다.', 0.3530300254004548), ('제83조 대통령은 국무총리·국무위원·행정각부의 장 기타 법률이 정하는 공사의 직을 겸할 수 없다.', 0.34266586524873427), ('제4장 정부\n제1절 대통령', 0.3292107244070076)]
[#0] 0.3535967762969585
제94조 행정각부의 장은 국무위원 중에서 국무총리의 제청으로 대통령이 임명한다.
[#1] 0.3530300254004548
제86조 ① 국무총리는 국회의 동의를 얻어 대통령이 임명한다.
② 국무총리는 대통령을 보좌하며, 행정에 관하여 대통령의 명을 받아 행정각부를 통할한다.
③ 군인은 현역을 면한 후가 아니면 국무총리로 임명될 수 없다.
[#2] 0.34266586524873427
제83조 대통령은 국무총리·국무위원·행정각부의 장 기타 법률이 정하는 공사의 직을 겸할 수 없다.
[#3] 0.3292107244070076
제4장 정부
제1절 대통령
ai >> 대한민국 헌법에 따르면 대통령이 되기 위해서는 국무총리로 임명되어 대통령을 보좌하고, 행정에 관하여 대통령의 명을 받아 행정각부를 통할 수 있는 국무총리가 되어야 합니다. 국무총리는 국회의 동의를 얻어 대통령이 임명하며, 군인은 현역을 면한 후가 아니면 국무총리로 임명될 수 없습니다. 따라서 닥터윌이 대통령이 되기 위해서는 먼저 국무총리가 되는 것이 필요합니다.
user> 대통령의 임기는?
[('제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.', 0.5381746278316393), ('제105조 ①